In [ ]:
!pip -q install fastapi uvicorn pyngrok python-multipart PyMuPDF pillow transformers accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 17.7 MB/s eta 0:00:00


In [ ]:
import io
import re
import threading
from typing import List

import fitz
import torch
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from pyngrok import ngrok
import uvicorn


# =========================
# الإعدادات
# =========================
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
USE_4BIT = False
DEFAULT_DPI = 200

CONTRACT_EXTRACTION_PROMPT = """
You are an advanced OCR system specialized in Arabic and English legal contracts.

مهمتك الوحيدة:
Extract ALL text exactly as shown in the image with maximum accuracy.

Rules / القواعد:

1. Extract everything exactly:
- Arabic text
- English text
- Numbers
- Dates
- Names
- Addresses
- Currency amounts
- Article numbers
- Tables

2. Formatting:
- Keep original reading order
- Arabic from right to left
- Preserve paragraphs
- Put titles in separate lines
- Tables as:
column1 | column2 | column3

3. Never:
- Do not explain
- Do not summarize
- Do not rewrite
- Do not translate
- Do not add comments

4. If unclear:
Use [غير واضح] or [unclear]

Start extraction now.
"""


# =========================
# أدوات مساعدة
# =========================
def resize_for_vlm(img: Image.Image, max_pixels: int = 1280 * 1280) -> Image.Image:
    w, h = img.size
    current_pixels = w * h

    if current_pixels > max_pixels:
        scale = (max_pixels / current_pixels) ** 0.5
        new_w = int(w * scale)
        new_h = int(h * scale)
        img = img.resize((new_w, new_h), Image.LANCZOS)

    return img


def clean_extracted_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(lines).strip()


def pdf_to_images(file_bytes: bytes, dpi: int = DEFAULT_DPI) -> List[Image.Image]:
    images = []
    doc = fitz.open(stream=file_bytes, filetype="pdf")

    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)

    for page_num in range(len(doc)):
        page = doc[page_num]
        pixmap = page.get_pixmap(matrix=matrix, alpha=False)
        img_bytes = pixmap.tobytes("png")
        pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        images.append(pil_img)

    doc.close()
    return images


def load_image(file_bytes: bytes) -> List[Image.Image]:
    img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    return [img]


# =========================
# تحميل المودل
# =========================
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(f"\n⏳ Loading model: {MODEL_ID}")

if USE_4BIT:
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("✅ Model loaded successfully.")


# =========================
# استخراج من صورة واحدة
# =========================
def extract_page(img: Image.Image) -> str:
    img = resize_for_vlm(img)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": CONTRACT_EXTRACTION_PROMPT}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.0,
            do_sample=False,
            repetition_penalty=1.05,
        )

    generated = output_ids[:, inputs.input_ids.shape[1]:]
    result = processor.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return clean_extracted_text(result[0])


# =========================
# FastAPI
# =========================
app = FastAPI(title="Contract OCR API")


@app.get("/")
def root():
    return {"message": "Contract OCR API is running"}


@app.post("/extract")
async def extract(file: UploadFile = File(...)):
    try:
        filename = file.filename.lower()
        file_bytes = await file.read()

        if filename.endswith(".pdf"):
            images = pdf_to_images(file_bytes)
        elif filename.endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff", ".tif")):
            images = load_image(file_bytes)
        else:
            return JSONResponse(
                status_code=400,
                content={
                    "success": False,
                    "error": "Unsupported file type. Only PDF and images are allowed."
                }
            )

        pages = []
        full_text_parts = []

        for i, img in enumerate(images, start=1):
            text = extract_page(img)
            pages.append({
                "page": i,
                "text": text
            })
            full_text_parts.append(text)

        full_text = "\n\n".join(full_text_parts).strip()

        return {
            "success": True,
            "page_count": len(images),
            "full_text": full_text,
            "pages": pages,
            "model": MODEL_ID
        }

    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={
                "success": False,
                "error": str(e)
            }
        )


# =========================
# تشغيل السيرفر + ngrok
# =========================
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)


thread = threading.Thread(target=run)
thread.start()

GPU available: True
GPU: Tesla T4

⏳ Loading model: Qwen/Qwen2.5-VL-3B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded successfully.


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

INFO:     Started server process [580]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Selecting previously unselected package cloudflared.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!cloudflared tunnel --url http://localhost:8000

2026-04-15T14:58:40Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-15T14:58:40Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-15T14:58:43Z INF +--------------------------------------------------------------------------------------------+
2026-04-15T14:58:43Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-15T14:58:43Z INF |  https://produces-authentic-simon-steady.trycloudflare

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     205.169.39.6:0 - "GET / HTTP/1.1" 200 OK
INFO:     34.123.170.104:0 - "GET / HTTP/1.1" 200 OK
INFO:     34.123.170.104:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     34.122.147.229:0 - "GET / HTTP/1.1" 200 OK
INFO:     205.169.39.29:0 - "GET / HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37.224.52.18:0 - "POST /extract HTTP/1.1" 200 OK
INFO:     37

In [ ]:
  ngrok.set_auth_token("YOUR_NGROK_TOKEN")
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

INFO:     Started server process [791]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


ERROR:pyngrok.process.ngrok:t=2026-04-15T14:37:30+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T14:37:30+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T14:37:30+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nY

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.